# Environment Setup

In [1]:
!pip -q install datasets==3.6.0
!pip -q install -U huggingface_hub hf_transfer
!pip -q install -U duckdb huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 2.5 MB/s eta 0:00:00


In [56]:
# === Standard Library Imports ===
import csv
import gzip
import hashlib
import io
import json
import math
import os
import re
import tarfile
import threading
import time
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import islice
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
from urllib.parse import urlparse

# === Third-Party Library Imports ===
import duckdb
import gdown
import numpy as np
import pandas as pd
import requests
import torch
from datasets import load_dataset
from huggingface_hub import HfApi, list_repo_files, login
from PIL import Image, UnidentifiedImageError
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm
from urllib3.util import Retry
from requests.adapters import HTTPAdapter

# === Google Colab Specific Imports ===
# (Only works in a Google Colab environment)
try:
    from google.colab import drive, userdata
except ImportError:
    print("Google Colab specific libraries not found. Skipping import.")

In [3]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Retrieve the token from Colab's Secrets Manager
hf_token = userdata.get('HF_TOKEN')

# Log in to HF using the retrieved token
login(hf_token)

In [4]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# Working with Hugging Face Datasets

### Downloading files using HF datasets

In [ ]:
# Dwnloading the data from hugging face datasets.
reviews = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Clothing_Shoes_and_Jewelry", trust_remote_code=True)
items = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Clothing_Shoes_and_Jewelry", split="full", trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading dataset shards:   0%|          | 0/38 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

In [ ]:
print(reviews["full"])
print(items["full"])

Dataset({
    features: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase'],
    num_rows: 66033346
})

### Counting the Non-Null values in each column.

In [ ]:
def count_non_null(batch, columns):
    """Count non-null values in specified columns for a batch"""
    counts = {col: [] for col in columns}
    batch_size = len(batch[list(batch.keys())[0]]) # Get the size of the batch

    for i in range(batch_size):
        for col in columns:
            if col in batch:
                # Check if the value for the current example and column is not None
                # IN CASE OF PRICE COLUMN IT IS 'None' STRING and in case of features and descriptions it is '[]' empty list,
                # so do the appropriate changes accordingly in below code
                counts[col].append(1 if batch[col][i] is not None else 0)
            else:
                # If column not in batch, append 0 for this example
                counts[col].append(0)
    return counts

# Columns to check
columns = ['average_rating', 'rating_number']

# Count non-null values with multiprocessing
results = items.map(
    count_non_null,
    fn_kwargs={'columns': columns},
    batched=True,
    batch_size=1000,
    num_proc=4,  # Use 4 processes
    remove_columns=items.column_names  # Remove original columns to save memory
)

# Aggregate results
final_counts = {col: sum(results[col]) for col in columns}

# Print results
print("Non-null value counts:")
for col, count in final_counts.items():
    print(f"- {col}: {count:,} ({(count/len(items))*100:.1f}%)")

Map (num_proc=4):   0%|          | 0/7218481 [00:00<?, ? examples/s]

Non-null value counts:
- average_rating: 7,218,481.0 (100.0%)
- rating_number: 7,218,481 (100.0%)


In [ ]:
# NOT RECOMMENDED: This is a very bad way to check null values as the RAM will spike a lot.
column_name = 'helpful_vote'
x =reviews['full'][column_name]
print(x[:10])
add_ = 0
for i in x:
  if i == 0:
    add_ += 1
print(add_)

### Benchmarking the performance of HF datasets and DuckDB

In [ ]:
# To benchmark the difference in execution time of HF datasets and DuckDB!!!!
NPROC = min(8, os.cpu_count() or 2)

t0 = time.perf_counter()
# Filter keeps only verified rows (runs in parallel on CPU)
verified_ds = reviews.filter(lambda x: bool(x["verified_purchase"]), num_proc=NPROC)
count_ds = verified_ds.num_rows['full']
t1 = time.perf_counter()

print(f"[datasets] verified count = {count_ds:,}  | time = {t1 - t0:.2f}s  | num_proc={NPROC}")

In [ ]:
# TO use DuckDB we need to first convert the required data to parquet format for maximum efficiency.
parquet_dir = "/content/reviews_parquet"

# Keep only what we need for this quick benchmark to keep files tiny
need_cols = [c for c in reviews['full'].column_names if c in ("verified_purchase",)]
reviews_small = reviews['full'].remove_columns([c for c in reviews['full'].column_names if c not in need_cols])

# Write shards to Parquet (this creates multiple files under the folder)
reviews_small.to_parquet(parquet_dir)

In [ ]:
parquet_path = "/content/reviews_parquet"  # <-- use YOUR actual file path

con = duckdb.connect()
con.execute(f"PRAGMA threads={min(8, os.cpu_count() or 2)};")
con.execute("PRAGMA memory_limit='8GB';")

t0 = time.perf_counter()
count_duck = con.execute("""
  SELECT COUNT(*)
  FROM read_parquet(?)
  WHERE verified_purchase = TRUE
""", [parquet_path]).fetchone()[0]
t1 = time.perf_counter()

print(f"[duckdb ] verified count = {count_duck:,} | time = {t1 - t0:.2f}s")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[duckdb ] verified count = 62,175,766 | time = 2.16s


***CLEARLY DUCKDB IS BLAZINGLY FAST !!!***

### Preprocessing before converting to parquet.

In [ ]:
# REVIEWS — keep slim columns, filter verified
rev_keep = ["user_id","parent_asin","timestamp","rating","verified_purchase","helpful_vote"]
reviews_slim = reviews["full"].remove_columns([c for c in reviews["full"].column_names if c not in rev_keep])
reviews_slim = reviews_slim.filter(lambda x: bool(x["verified_purchase"]), num_proc=4)
# (Optional) drop the flag now that you’ve filtered:
reviews_slim = reviews_slim.remove_columns(["verified_purchase"])

Filter (num_proc=4):   0%|          | 0/66033346 [00:00<?, ? examples/s]

In [ ]:
# ITEMS — extract only what you need; we’ll map images→main_image_url later
itm_keep = ["parent_asin","main_category","title","average_rating","rating_number","price","images","categories","features","description","categories","details"]
items_slim = items.remove_columns([c for c in items.column_names if c not in itm_keep])

### Extracting the Main Image URL

In [ ]:
NPROC = min(8, os.cpu_count() or 2)

def _first_url(val):
    """Return the first non-empty string URL found inside val (str/list/ndarray/dict)."""
    if val is None:
        return None
    if isinstance(val, str):
        s = val.strip()
        return s or None
    if isinstance(val, (list, tuple, np.ndarray)):
        for x in val:
            u = _first_url(x)
            if u:
                return u
        return None
    if isinstance(val, dict):
        # common keys in the Amazon dumps; prioritize hi_res -> large -> medium -> url
        for k in ("hi_res", "large", "thumb"):
            if k in val:
                u = _first_url(val[k])
                if u:
                    return u
        return None
    # anything else
    return None

def to_main_url(batch):
    """datasets.map(batched=True) callback: reads batch['images'] (dicts) -> main_image_url"""
    out = []
    for img in batch["images"]:
        url = None
        if isinstance(img, dict):
            url = _first_url(img)           # dict case (your schema)
        else:
            url = _first_url(img)           # be tolerant to accidental list/str formats
        out.append(url)
    batch["main_image_url"] = out
    return batch


In [ ]:
# items_slim must contain the 'images' column
items_slim = items_slim.map(to_main_url, batched=True, num_proc=NPROC)
items_slim = items_slim.remove_columns(["images"])

In [ ]:
# Checking how many products there are without ant image.
len(items_slim.filter(lambda x: x["main_image_url"] is None, num_proc=NPROC))

### Converting to parquet format and saving locally and in google drive

In [ ]:
SAVE_DIR = '/content/artifacts/Post_HF_datasets'
os.makedirs(SAVE_DIR, exist_ok=True)

rev_path = f"{SAVE_DIR}/reviews_small_unpart"
itm_path = f"{SAVE_DIR}/items_small_unpart"

reviews_slim.to_parquet(rev_path)

size_bytes = os.path.getsize(rev_path)

def human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024:
            return f"{n:.2f} {u}"
        n /= 1024
print("size:", human(size_bytes))

items_slim.to_parquet(itm_path)

Creating parquet from Arrow format:   0%|          | 0/62176 [00:00<?, ?ba/s]

4352307484

In [ ]:
# SAVE_DIR = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_HF_datasets'
# os.makedirs(SAVE_DIR, exist_ok=True)

# rev_g = f'{SAVE_DIR}/reviews_small_unpart'
# itm_g = f'{SAVE_DIR}/items_small_unpart'

# # write with a proper extension + compression
# reviews_slim.to_parquet(rev_g, engine='pyarrow', compression='snappy', index=False)
# items_slim.to_parquet(itm_g, engine='pyarrow', compression='snappy', index=False)

# # quick check
# import os
# print('reviews:', os.path.getsize(rev_g)/1024**2, 'MB')
# print('items  :', os.path.getsize(itm_g)/1024**2, 'MB')

# Downloading the parquet file from Google Drive Link

In [ ]:
# Your shared links (file IDs extracted below)
REV_ID = "1xTxaWEYpCxIVJPwprG4NFUL02SyRWQsn"
ITM_ID = "155CP80vSsf2CNp3DUlXQ_A9EgMAvL_Wq"

SAVE_DIR = '/content/artifacts/Post_HF_datasets'
os.makedirs(SAVE_DIR, exist_ok=True)

REV_OUT = f'{SAVE_DIR}/reviews_small_unpart'
ITM_OUT = f'{SAVE_DIR}/items_small_unpart'

gdown.download(id=REV_ID, output=REV_OUT, quiet=False)
gdown.download(id=ITM_ID, output=ITM_OUT, quiet=False)

# (optional) show sizes
def human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024: return f"{n:.2f} {u}"
        n /= 1024
    return f"{n:.2f} PB"

print("reviews size:", human(os.path.getsize(REV_OUT)))
print("items   size:", human(os.path.getsize(ITM_OUT)))

Downloading...
From (original): https://drive.google.com/uc?id=1xTxaWEYpCxIVJPwprG4NFUL02SyRWQsn
From (redirected): https://drive.google.com/uc?id=1xTxaWEYpCxIVJPwprG4NFUL02SyRWQsn&confirm=t&uuid=f1dcb80b-1a8d-4b99-89ea-1d047a635e41
To: /content/artifacts/Post_HF_datasets/reviews_small_unpart
100%|██████████| 1.91G/1.91G [00:24<00:00, 78.2MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=155CP80vSsf2CNp3DUlXQ_A9EgMAvL_Wq
From (redirected): https://drive.google.com/uc?id=155CP80vSsf2CNp3DUlXQ_A9EgMAvL_Wq&confirm=t&uuid=3b9fdca4-bcc6-45d7-b5c1-038dbba32d51
To: /content/artifacts/Post_HF_datasets/items_small_unpart
100%|██████████| 4.20G/4.20G [00:46<00:00, 91.3MB/s]

reviews size: 1.78 GB
items   size: 3.91 GB


In [ ]:
st = time.time()
# Load the parquet files into datasets
reviews = load_dataset("parquet", data_files=REV_OUT)
items = load_dataset("parquet", data_files=ITM_OUT)

et = time.time()
print("Total execution time:", et - st, "seconds")
# You might want to inspect the loaded datasets
print("Reviews dataset:")
print(reviews)
print("\nItems dataset:")
print(items)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

Total execution time: 108.02244067192078 seconds
Reviews dataset:
DatasetDict({
    train: Dataset({
        features: ['rating', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote'],
        num_rows: 62175766
    })
})

Items dataset:
DatasetDict({
    train: Dataset({
        features: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'categories', 'details', 'parent_asin', 'main_image_url'],
        num_rows: 7218481
    })
})


# Loading and Uploading data using HF

In [ ]:
api = HfApi()

repo_id = "PirateKing0402/Amazon_dataset"   # change this
api.create_repo(repo_id=repo_id, repo_type="dataset", private=False, exist_ok=True)

RepoUrl('https://huggingface.co/datasets/PirateKing0402/Amazon_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='PirateKing0402/Amazon_dataset')

### Uploading data to HF

In [ ]:
api.upload_file(
    path_or_fileobj=REV_OUT,
    path_in_repo="reviews_small_unpart",
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Add reviews parquet unpartitioned (hf_transfer)"
)

api.upload_file(
    path_or_fileobj=ITM_OUT,
    path_in_repo="items_small_unpart",
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Add items parquet unpartitioned (hf_transfer)"
)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /content/reviews_small_unpart         :   0%|          |  544kB / 1.91GB            

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /content/items_small_unpart           :   0%|          |  525kB / 4.20GB            

CommitInfo(commit_url='https://huggingface.co/datasets/PirateKing0402/Amazon_dataset/commit/a60281c9ad533dffaf04fc91a0b72f7ec7c29480', commit_message='Add items parquet unpartitioned (hf_transfer)', commit_description='', oid='a60281c9ad533dffaf04fc91a0b72f7ec7c29480', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/PirateKing0402/Amazon_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='PirateKing0402/Amazon_dataset'), pr_revision=None, pr_num=None)

### Loading data from HF
( Slower than Google Drive download, better to use DuckDB to get data from HF )

In [ ]:
# Remove the reviews and items variables from memory
del reviews
del items

# You can optionally add print statements to confirm they are deleted (will raise NameError if successful)
# print(reviews)
# print(items)

In [ ]:
from datasets import load_dataset

# Define the repository ID and file paths within the repo
repo_id = "PirateKing0402/Amazon_dataset"
reviews_file_path = "reviews_small_unpart"
items_file_path = "items_small_unpart"
st = time.time()
# Load the parquet files into datasets
reviews = load_dataset("parquet", data_files=f"hf://datasets/{repo_id}/{reviews_file_path}")
items = load_dataset("parquet", data_files=f"hf://datasets/{repo_id}/{items_file_path}")
et = time.time()

print("Total execution time:", et - st, "seconds")
# You might want to inspect the loaded datasets
print("Reviews dataset:")
print(reviews)
print("\nItems dataset:")
print(items)

reviews_small_unpart:   0%|          | 0.00/1.91G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

items_small_unpart:   0%|          | 0.00/4.20G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

Total execution time: 969.474189043045 seconds
Reviews dataset:
DatasetDict({
    train: Dataset({
        features: ['rating', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote'],
        num_rows: 62175766
    })
})

Items dataset:
DatasetDict({
    train: Dataset({
        features: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'categories', 'details', 'parent_asin', 'main_image_url'],
        num_rows: 7218481
    })
})


# Processing using DuckDB

### Using DuckDB on a data accessed through Remote Connection.
DuckDB's power lies in its ability to query massive remote datasets efficiently without local downloads. It achieves this by making many small, precise HTTP range requests to read only the parts of a file it needs.

This method is perfectly suited for cloud object stores like Amazon S3 or GCS, which are designed for this high-throughput access pattern. However, it triggers the anti-bot rate limits on standard web servers like the Hugging Face Hub, causing the query to fail.

In [ ]:
# ---- config: your repo + paths (file OR folder)
REPO_ID = "PirateKing0402/Amazon_dataset"
REV_PREFIX = "reviews_small_unpart"  # e.g. "reviews_small_unpart.parquet" OR a folder "reviews_small_unpart/"
ITM_PREFIX = "items_small_unpart"    # same idea for items

# Helper: build HTTPS URLs to the exact parquet files in the repo
def hf_parquet_urls(repo_id: str, prefix: str):
    files = list_repo_files(repo_id, repo_type="dataset")
    # case 1: a single file like "<prefix>.parquet"
    exact = [p for p in files if p == f"{prefix}"]
    if exact:
        return [f"https://huggingface.co/datasets/{repo_id}/resolve/main/{exact[0]}"]

rev_urls = hf_parquet_urls(REPO_ID, REV_PREFIX)
itm_urls = hf_parquet_urls(REPO_ID, ITM_PREFIX)

# Safety check: make sure we found files
print("review files:", len(rev_urls))
print("item files  :", len(itm_urls))
assert rev_urls, "No review parquet found in the repo/path you provided."
assert itm_urls, "No item parquet found in the repo/path you provided."

review files: 1
item files  : 1


In [ ]:
# ---- DuckDB setup (HTTP range reads; only needed bytes fetched)
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='8GB';")  # optional

# ---- Items: count duplicate rows by parent_asin
sql_items = """
WITH g AS (
  SELECT parent_asin, COUNT(*) AS cnt
  FROM read_parquet($urls)
  GROUP BY parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,        -- matches pandas .duplicated(...).sum()
  COUNT(*) AS keys_with_duplicates  -- number of parent_asin groups that had dupes
FROM g;
"""
items_dup = con.execute(sql_items, {"urls": itm_urls}).fetchdf()

In [ ]:
sql_reviews = """
WITH g AS (
  SELECT user_id, parent_asin, COUNT(*) AS cnt
  FROM read_parquet($urls)
  GROUP BY user_id, parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,            -- matches pandas .duplicated(...).sum()
  COUNT(*)                    AS keys_with_duplicates      -- number of (user,item) pairs that had dupes
FROM g;
"""
reviews_dup = con.execute(sql_reviews, {"urls": rev_urls}).fetchdf()

print("\nItems duplicates (pandas equivalence): duplicated(subset=['parent_asin']).sum()")
print(int(items_dup.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(items_dup.loc[0, "keys_with_duplicates"]))

print("\nReviews duplicates (pandas equivalence): duplicated(subset=['user_id','parent_asin']).sum()")
print(int(reviews_dup.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(reviews_dup.loc[0, "keys_with_duplicates"]))

### Using DuckDB on locally downloaded data.

#### Handling Duplicates

In [ ]:
# ---- DuckDB setup (using local files)
con = duckdb.connect()
con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='40GB';")  # optional

# Define local file paths
REV_PATH_LOCAL = "/content/artifacts/Post_HF_datasets/reviews_small_unpart"
ITM_PATH_LOCAL = "/content/artifacts/Post_HF_datasets/items_small_unpart"

In [ ]:
# ---- Items: count duplicate rows by parent_asin using local file
sql_items_local = """
WITH g AS (
  SELECT parent_asin, COUNT(*) AS cnt
  FROM read_parquet(?)
  GROUP BY parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,        -- matches pandas .duplicated(...).sum()
  COUNT(*) AS keys_with_duplicates  -- number of parent_asin groups that had dupes
FROM g;
"""
items_dup_local = con.execute(sql_items_local, [ITM_PATH_LOCAL]).fetchdf()

print("Items duplicates (local file):")
print(int(items_dup_local.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(items_dup_local.loc[0, "keys_with_duplicates"]))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Items duplicates (local file):
0 | keys_with_duplicates: 0


In [ ]:
# ---- Reviews: count duplicate rows by user_id and parent_asin using local file
sql_reviews_local = """
WITH g AS (
  SELECT user_id, parent_asin, COUNT(*) AS cnt
  FROM read_parquet(?)
  GROUP BY user_id, parent_asin
  HAVING COUNT(*) > 1
)
SELECT
  COALESCE(SUM(cnt - 1), 0) AS duplicate_rows,            -- matches pandas .duplicated(...).sum()
  COUNT(*)                    AS keys_with_duplicates      -- number of (user,item) pairs that had dupes
FROM g;
"""
reviews_dup_local = con.execute(sql_reviews_local, [REV_PATH_LOCAL]).fetchdf()

print("\nReviews duplicates (local file):")
print(int(reviews_dup_local.loc[0, "duplicate_rows"]),
      "| keys_with_duplicates:", int(reviews_dup_local.loc[0, "keys_with_duplicates"]))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Reviews duplicates (local file):
803670 | keys_with_duplicates: 684012


In [ ]:
# Using DuckDB on locally downloaded data to count rows with missing timestamps
sql_reviews_null_timestamp = """
WITH t AS (
  SELECT TRY_CAST("timestamp" AS BIGINT) AS ts
  FROM read_parquet(?)
)
SELECT COUNT(*) FROM t WHERE ts IS NULL;
"""
reviews_null_timestamp_count = con.execute(sql_reviews_null_timestamp, [REV_PATH_LOCAL]).fetchone()[0]

print(f"Number of rows with null timestamp in reviews (local file): {reviews_null_timestamp_count}")

Number of rows with null timestamp in reviews (local file): 0


In [ ]:
REV_PATH_LOCAL = "/content/artifacts/Post_HF_datasets/reviews_small_unpart"
ITM_PATH_LOCAL = "/content/artifacts/Post_HF_datasets/items_small_unpart"

SAVE_DIR = '/content/artifacts/Post_dedup'
os.makedirs(SAVE_DIR, exist_ok=True)

REV_DEDUP_LOCAL = f"{SAVE_DIR}/reviews_small_unpart"          # <-- output file to create
ITM_DEDUP_LOCAL = f"{SAVE_DIR}/items_small_unpart"

os.makedirs(os.path.dirname(REV_DEDUP_LOCAL), exist_ok=True)
os.makedirs(os.path.dirname(ITM_DEDUP_LOCAL), exist_ok=True)

# 0) how many rows before
orig_cnt = con.execute("SELECT COUNT(*) FROM read_parquet(?)", [REV_PATH_LOCAL]).fetchone()[0]
print("rows before:", orig_cnt)

rows before: 62175766


In [ ]:
# 1) write a deduplicated copy:
#    - partition by (user_id, parent_asin)
#    - order by timestamp DESC so we keep the newest
#    - tie-breakers: helpful_vote DESC, rating DESC (optional but sensible)
#    - keep exactly one row: rn = 1
def sql_quote(path: str) -> str:
    # escape any single quotes in a file path for SQL
    return path.replace("'", "''")

src = sql_quote(REV_PATH_LOCAL)
dst = sql_quote(REV_DEDUP_LOCAL)

sql = f"""
COPY (
  WITH raw AS (
    SELECT
      user_id,
      parent_asin,
      TRY_CAST("timestamp" AS BIGINT) AS ts,  -- keep safe quoting
      rating,
      helpful_vote
    FROM read_parquet('{src}')
  ),
  ranked AS (
    SELECT
      *,
      ROW_NUMBER() OVER (
        PARTITION BY user_id, parent_asin
        ORDER BY ts DESC NULLS LAST, helpful_vote DESC NULLS LAST, rating DESC NULLS LAST
      ) AS rn
    FROM raw
  )
  SELECT user_id, parent_asin, ts AS "timestamp", rating, helpful_vote
  FROM ranked
  WHERE rn = 1
) TO '{dst}'
  (FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(sql)

# 2) check after
dedup_cnt = con.execute("SELECT COUNT(*) FROM read_parquet(?)", [REV_DEDUP_LOCAL]).fetchone()[0]
print("rows after :", dedup_cnt)
print("removed    :", orig_cnt - dedup_cnt)

# Copy the items data from the local path to the destination path
# Although we did not perform deduplication on items data,
# we are copying it to the Post_dedup folder for consistency with the reviews data.
itm_src = sql_quote(ITM_PATH_LOCAL)
itm_dst = sql_quote(ITM_DEDUP_LOCAL)

sql_copy_items = f"""
COPY (
  SELECT * FROM read_parquet('{itm_src}')
) TO '{itm_dst}'
  (FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(sql_copy_items)

print(f"Items data copied from {ITM_PATH_LOCAL} to {ITM_DEDUP_LOCAL}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows after : 61372096
removed    : 803670


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Items data copied from /content/artifacts/Post_HF_datasets/items_small_unpart to /content/artifacts/Post_dedup/items_small_unpart


In [ ]:
SAVE_DIR = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_dedup'
os.makedirs(SAVE_DIR, exist_ok=True)

rev_g = f'{SAVE_DIR}/reviews_small_unpart'
itm_g = f'{SAVE_DIR}/items_small_unpart'

sql_copy_reviews = f"""
COPY (SELECT * FROM read_parquet('{REV_DEDUP_LOCAL}'))
TO '{rev_g}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_reviews)

sql_copy_items = f"""
COPY (SELECT * FROM read_parquet('{ITM_DEDUP_LOCAL}'))
TO '{itm_g}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_items)

# quick check
import os
print('reviews:', os.path.getsize(rev_g)/1024**2, 'MB')
print('items  :', os.path.getsize(itm_g)/1024**2, 'MB')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

reviews: 2541.530979156494 MB
items  : 3953.9629278182983 MB


#### Pre-necessities

In [ ]:
# Define the file ID from the shared link
file_id1 = "1_tBi5CUppymiD-ZLgazZcFFhjWSx1PrS"
file_id2 = "1kZXUItfGCKyIGqhYBSWG9lUR1Uu6jj3p"

# Define the local path to save the downloaded file
SAVE_DIR = '/content/artifacts/Working'
os.makedirs(SAVE_DIR, exist_ok=True)
local_file_path1 = f'{SAVE_DIR}/reviews' # You can rename the file as needed
local_file_path2 = f'{SAVE_DIR}/items'

# Download the file
print(f"Downloading file with ID: {file_id1} to {local_file_path1}")
gdown.download(id=file_id1, output=local_file_path1, quiet=False)

# Download the file
print(f"Downloading file with ID: {file_id2} to {local_file_path2}")
gdown.download(id=file_id2, output=local_file_path2, quiet=False)


Downloading...
From (original): https://drive.google.com/uc?id=1_tBi5CUppymiD-ZLgazZcFFhjWSx1PrS
From (redirected): https://drive.google.com/uc?id=1_tBi5CUppymiD-ZLgazZcFFhjWSx1PrS&confirm=t&uuid=2ea9b8c0-cbc5-45b2-987b-0a0d74c257b6
To: /content/artifacts/Working/reviews
100%|██████████| 2.66G/2.66G [00:37<00:00, 71.1MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1kZXUItfGCKyIGqhYBSWG9lUR1Uu6jj3p
From (redirected): https://drive.google.com/uc?id=1kZXUItfGCKyIGqhYBSWG9lUR1Uu6jj3p&confirm=t&uuid=e3e6d133-1ab3-4058-a820-86fcfeb100b5
To: /content/artifacts/Working/items
100%|██████████| 4.15G/4.15G [01:14<00:00, 55.5MB/s]


'/content/artifacts/Working/items'

In [ ]:
# Establish DuckDB connection
con = duckdb.connect()
con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='40GB';")  # optional

#### Extracting the Main Image URL

In [ ]:
# Execute the query and fetch all results into a Python list
initial_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{local_file_path2}')").fetchone()[0]
print(initial_count)

7218481


In [ ]:
# Filter items data to remove rows where main_image_url is null using DuckDB

ITM_FILTERED_LOCAL = f"{SAVE_DIR}/items_small_filtered"

# SQL query to filter and copy the data
sql_filter_items = f"""
COPY (
  SELECT *
  FROM read_parquet('{local_file_path2}')
  WHERE main_image_url IS NOT NULL
) TO '{ITM_FILTERED_LOCAL}'
  (FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(sql_filter_items)

# Quick check of the number of rows after filtering
filtered_items_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{ITM_FILTERED_LOCAL}')").fetchone()[0]
print(f"Number of rows in items data after filtering for main_image_url: {filtered_items_count}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Number of rows in items data after filtering for main_image_url: 7208433


In [ ]:
temp4 = con.execute(f"SELECT main_image_url from read_parquet('{ITM_FILTERED_LOCAL}')").fetchall()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
for item in temp4:
  # The URL is the first element of the tuple
  url = item[0] if isinstance(item, tuple) and len(item) > 0 else "no no no"
  if url and isinstance(url, str) and url.startswith('http'):
    pass
  else:
    print(f"Found a URL that does not start with 'http' or is not a valid URL: {url}")
    print(type(url))

print("Finished checking URLs.")

Finished checking URLs.


#### Extracting the gender column

In [ ]:
items_with_gender_clean = "/content/artifacts/Working/items_with_gender_clean"

os.makedirs(os.path.dirname(items_with_gender_clean), exist_ok=True)


sql = rf"""
COPY (
  WITH src AS (
    SELECT * FROM read_parquet('{ITM_FILTERED_LOCAL}')
  ),
  x AS (
    SELECT
      *,
      regexp_extract(CAST(details AS VARCHAR),
                     '(?i)"department"\s*:\s*"([^"]+)"', 1) AS gender_raw
    FROM src
    WHERE regexp_matches(CAST(details AS VARCHAR),
                         '(?i)"department"\s*:\s*"')
  ),
  norm AS (
    SELECT
      *,
      lower(
        regexp_replace(
          regexp_replace(gender_raw, '[^A-Za-z -]+', ''),   -- keep letters / space / hyphen
          '\s+', ' '                                       -- collapse whitespace
        )
      ) AS dep_norm
    FROM x
  ),
  map AS (
    SELECT
      *,
      CASE
        -- specific buckets first
        WHEN dep_norm LIKE '%baby%girl%' OR dep_norm LIKE '%girl%baby%' THEN 'baby-girls'
        WHEN dep_norm LIKE '%baby%boy%'  OR dep_norm LIKE '%boy%baby%'  THEN 'baby-boys'
        WHEN dep_norm LIKE '%unisex%baby%'                               THEN 'unisex-baby'
        WHEN dep_norm LIKE '%unisex%child%'                              THEN 'unisex-child'
        WHEN dep_norm LIKE '%unisex%adult%'                              THEN 'unisex-adult'
        -- canonicalize women/men to womens/mens
        WHEN dep_norm LIKE '%womens%' OR dep_norm LIKE '%women%'         THEN 'womens'
        WHEN dep_norm LIKE '%mens%'   OR dep_norm LIKE '%men%'           THEN 'mens'
        WHEN dep_norm LIKE '%girls%'  OR dep_norm LIKE '%girl%'          THEN 'girls'
        WHEN dep_norm LIKE '%boys%'   OR dep_norm LIKE '%boy%'           THEN 'boys'
        ELSE NULL
      END AS gender
    FROM norm
  ),
  kept AS (
    SELECT *
    FROM map
    WHERE gender IN (
      'womens','mens','girls','boys',
      'baby-girls','baby-boys',
      'unisex-adult','unisex-child','unisex-baby'
    )
  )
  SELECT * EXCLUDE (dep_norm, gender_raw) FROM kept
) TO '{items_with_gender_clean}' (FORMAT PARQUET, CODEC ZSTD);
"""

con.execute(sql)
print("Wrote:", items_with_gender_clean)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /content/artifacts/Working/items_with_gender_clean


In [ ]:
con.execute(f"SELECT count(*) from read_parquet('{items_with_gender_clean}')").fetchone()[0]

6271634

#### Extracting the dimensions and weights

In [ ]:
items_with_dimension_weight = "/content/artifacts/Working/items_with_dimension_weight"
os.makedirs(os.path.dirname(items_with_dimension_weight), exist_ok=True)

con = duckdb.connect()

# 0) Source view with a text copy
con.execute(f"""
CREATE OR REPLACE TEMP VIEW src AS
SELECT *, CAST(details AS VARCHAR) AS dtxt
FROM read_parquet('{items_with_gender_clean}');
""")
print("src count:", con.execute("SELECT COUNT(*) FROM src").fetchone()[0])

src count: 6271634


In [ ]:
# 1) Create the view (no fetch here)
con.execute(r"""
CREATE OR REPLACE TEMP VIEW dims5 AS
SELECT
  *,
  -- 1) Package Dimensions (we can keep it simple if we also extract item-package separately)
  regexp_extract(dtxt, '(?is)"package\s+dimensions"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1) AS dim_package,

  -- 2) Product Dimensions
  regexp_extract(dtxt, '(?is)"product\s+dimensions"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1) AS dim_product,

  -- 3) Item Package Dimensions L x W x H
  regexp_extract(dtxt, '(?is)"item\s+package\s+dimensions\s*l\s*x\s*w\s*x\s*h"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1) AS dim_item_pkg_lxwxh,

  -- 4 & 5) Item Dimensions LxWxH (handles one or many spaces before LxWxH)
  regexp_extract(dtxt, '(?is)"item\s+dimensions\s+lxwxh"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1) AS dim_item_lxwxh
FROM src;
""")



In [ ]:
con.execute(r"""
CREATE OR REPLACE TEMP VIEW prio AS
SELECT
  *,
  CASE
    WHEN dim_product        IS NOT NULL AND dim_product != '' THEN dim_product
    WHEN dim_item_lxwxh     IS NOT NULL AND dim_item_lxwxh != '' THEN dim_item_lxwxh
    WHEN dim_item_pkg_lxwxh IS NOT NULL AND dim_item_pkg_lxwxh != '' THEN dim_item_pkg_lxwxh
    WHEN dim_package        IS NOT NULL AND dim_package != '' THEN dim_package
    ELSE NULL
  END AS dimensions_raw
FROM dims5;
""")

In [ ]:
# Add weight (Item Weight → Package Weight), then build dimension_weight with CASE
con.execute(r"""
CREATE OR REPLACE TEMP VIEW with_weight AS
SELECT
  p.*,
  CASE
    WHEN regexp_matches(dtxt, '(?is)"item weight"[^:：]*[:：]\s*["“]')
      THEN regexp_extract(dtxt, '(?is)"item weight"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1)
    WHEN regexp_matches(dtxt, '(?is)"package weight"[^:：]*[:：]\s*["“]')
      THEN regexp_extract(dtxt, '(?is)"package weight"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1)
    ELSE NULL
  END AS weight_raw
FROM prio p;
""")

In [ ]:
con.execute(r"""
CREATE OR REPLACE TEMP VIEW final AS
SELECT
  * EXCLUDE (dtxt, dim_package, dim_product, dim_item_pkg_lxwxh, dim_item_lxwxh, dimensions_raw, weight_raw),
  CASE
    WHEN dimensions_raw IS NOT NULL AND weight_raw IS NOT NULL
      THEN dimensions_raw || '; ' || weight_raw
    WHEN dimensions_raw IS NOT NULL
      THEN dimensions_raw
    ELSE weight_raw
  END AS dimension_weight
FROM with_weight where dimension_weight is not null;
""")

con.execute(f"COPY (SELECT * FROM final) TO '{items_with_dimension_weight}' (FORMAT PARQUET, CODEC ZSTD)")
print("Wrote:", items_with_dimension_weight)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /content/artifacts/Working/items_with_dimension_weight


In [ ]:
con.execute(f"SELECT count(*) from read_parquet('{items_with_dimension_weight}')").fetchone()[0]

5114268

#### Extracting the Manufacturer/Brand

In [ ]:
items_with_manufacturer = "/content/artifacts/Working/items_with_manufacturer"

os.makedirs(os.path.dirname(items_with_manufacturer), exist_ok=True)

sql = rf"""
COPY (
  WITH src AS (
    SELECT * FROM read_parquet('{items_with_dimension_weight}')
  ),
  m AS (
    SELECT
      *,
      -- ONLY Manufacturer; tolerant to spacing, unicode colon, curly quotes
      NULLIF(
        trim(
          regexp_replace(
            regexp_extract(
              CAST(details AS VARCHAR),
              '(?is)"manufacturer"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1
            ),
            '\s+', ' '  -- collapse whitespace
          )
        ),
        ''
      ) AS manufacturer
    FROM src
  )
  SELECT *
  FROM m where manufacturer != ''
) TO '{items_with_manufacturer}' (FORMAT PARQUET, CODEC ZSTD);
"""

con.execute(sql)
print("Wrote:", items_with_manufacturer)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /content/artifacts/Working/items_with_manufacturer


In [ ]:
con.execute(f"SELECT count(*) from read_parquet('{items_with_manufacturer}')").fetchone()[0]

2683344

#### Extracting Main Category

In [ ]:
items_with_main_categories = "/content/artifacts/Working/items_with_main_categories"

os.makedirs(os.path.dirname(items_with_main_categories), exist_ok=True)

sql = f"""
COPY (
  WITH src AS (
    SELECT * FROM read_parquet('{items_with_manufacturer}')
  )
  SELECT
    * EXCLUDE(main_category),
    lower(
      list_element(
        list_filter(categories, x -> lower(x) IN ('clothing','shoes','jewelry')),
        1
      )
    ) AS main_categories
  FROM src where main_categories is not null
) TO '{items_with_main_categories}' (FORMAT PARQUET, CODEC ZSTD);
"""

con.execute(sql)
print("Wrote:", items_with_main_categories)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /content/artifacts/Working/items_with_main_categories


In [ ]:
con.execute(f"SELECT count(*) from read_parquet('{items_with_main_categories}')").fetchone()[0]

2045764

#### Extracting DFA (Date First Available)

In [ ]:
items_with_DFA = "/content/artifacts/Working/items_with_dfa"

os.makedirs(os.path.dirname(items_with_DFA), exist_ok=True)

sql = rf"""
COPY (
  WITH src AS (
    SELECT * FROM read_parquet('{items_with_main_categories}')
  ),
  dfa AS (
    SELECT
      *,
      -- ONLY "Date First Available"; tolerant to spacing, unicode colon, curly quotes
      NULLIF(
        trim(
          regexp_replace(
            regexp_extract(
              CAST(details AS VARCHAR),
              '(?is)"date\s+first\s+available"[^:：]*[:：]\s*["“]([^"”]+)["”]', 1
            ),
            '\s+', ' '  -- collapse whitespace
          )
        ),
        ''
      ) AS date_first_available
    FROM src where date_first_available != ''
  )
  -- keep schema, add the new column (you can drop 'details' if you don't need it)
  SELECT *
  FROM dfa
) TO '{items_with_DFA}' (FORMAT PARQUET, CODEC ZSTD);
"""

con.execute(sql)
print("Wrote:", items_with_DFA)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /content/artifacts/Working/items_with_dfa


In [ ]:
con.execute(f"SELECT count(*) from read_parquet('{items_with_DFA}')").fetchone()[0]

2032790

#### Extracting leaf category

In [ ]:
items_with_leaf_category = "/content/artifacts/Working/items_with_leaf_category"

os.makedirs(os.path.dirname(items_with_leaf_category), exist_ok=True) # Corrected variable name

sql = f"""
COPY (
  WITH src AS (
    SELECT * FROM read_parquet('{items_with_DFA}')
  )
  SELECT
    *,
    CASE
      WHEN categories IS NULL OR len(categories) = 0  -- Corrected function name
        THEN NULL
      ELSE
        lower(
          trim(
            list_element(categories, len(categories))  -- Corrected function name
          )
        )
    END AS leaf_category
  FROM src
) TO '{items_with_leaf_category}' (FORMAT PARQUET, CODEC ZSTD);
"""

con.execute(sql)
print("Wrote:", items_with_leaf_category)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /content/artifacts/Working/items_with_leaf_category


In [ ]:
SAVE_DIR = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_items_cleanup'
os.makedirs(SAVE_DIR, exist_ok=True)

itm_g = f'{SAVE_DIR}/items_cleaned'

sql_copy_items = f"""
COPY (SELECT * FROM read_parquet('{items_with_leaf_category}'))
TO '{itm_g}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_items)

# quick check
import os
print('items  :', os.path.getsize(itm_g)/1024**2, 'MB')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

items  : 1216.1520309448242 MB


#### Extracting reviews based on filtered items

In [ ]:
REVIEWS_OUT = "/content/artifacts/Working/reviews_filtered_by_items"

os.makedirs(os.path.dirname(REVIEWS_OUT), exist_ok=True)

# Source views
con.execute(f"CREATE OR REPLACE TEMP VIEW items_ok AS SELECT parent_asin FROM read_parquet('{items_with_leaf_category}');")
con.execute(f"CREATE OR REPLACE TEMP VIEW reviews_raw AS SELECT * FROM read_parquet('{local_file_path1}');")

# Filter with a simple INNER JOIN (you said items have no duplicates)
con.execute("""
CREATE OR REPLACE TEMP VIEW reviews_filtered AS
SELECT r.*
FROM reviews_raw r
JOIN items_ok i
  ON i.parent_asin = r.parent_asin;
""")

# Counts (before/after) using the filtered relation you asked for
n_before = con.execute("SELECT COUNT(*) FROM reviews_raw").fetchone()[0]
n_after  = con.execute("SELECT COUNT(*) FROM reviews_filtered").fetchone()[0]
print(f"Reviews before: {n_before:,} | after filter: {n_after:,} | dropped: {n_before - n_after:,}")

# Persist filtered reviews
con.execute(f"""
COPY (SELECT * FROM reviews_filtered)
TO '{REVIEWS_OUT}' (FORMAT PARQUET, CODEC ZSTD);
""")
print("Wrote:", REVIEWS_OUT)

# Optional peek
df4 = con.execute("SELECT * FROM reviews_filtered LIMIT 100").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Reviews before: 61,372,096 | after filter: 14,230,546 | dropped: 47,141,550


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /content/artifacts/Working/reviews_filtered_by_items


In [ ]:
df4.head()

,user_id,parent_asin,timestamp,rating,helpful_vote
0,AE226DXXSDWPBFTQB3M4VMOVZR2A,B07XFXXZMV,1631309290873,5.0,0
1,AE22ATMNSAFC5TQQ36NZZB2EYTVQ,B07FKVFNHC,1676746531768,5.0,0
2,AE22C2TILGPY7U23BJX2ZW4SVPMQ,B07VNRDVK3,1481071911000,5.0,0
3,AE22F4QOZJQ4N3ZUG7ZZUEOWHIJQ,B08DX4TJLQ,1647118774918,4.0,0
4,AE22GWOMEVMBAD7OWMX2KZQW756Q,B06XR38N19,1618352294318,4.0,0


#### Temporal Splitting

In [ ]:
TimestampLike = Union[pd.Timestamp, str, int, float]

def temporal_split_ms_duckdb(
    src_parquet: str,
    time_col: str = "ts",
    *,
    test_fraction: Optional[float] = None,
    cutoff: Optional[TimestampLike] = None,
    train_includes_cutoff: bool = True,
    drop_na_time: bool = True,
    sort_within_splits: bool = False,
    out_prefix: Optional[str] = "splits/v1"
) -> Tuple[int, pd.Timestamp, int, int]:
    if (test_fraction is None) == (cutoff is None):
        raise ValueError("Provide exactly one of `test_fraction` or `cutoff`.")
    if test_fraction is not None:
        if not np.isfinite(test_fraction) or not (0.0 < float(test_fraction) < 1.0):
            raise ValueError("`test_fraction` must be finite and in (0,1).")

    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE TEMP VIEW _raw AS SELECT * FROM read_parquet('{src_parquet}')")

    where_clause = ""
    if drop_na_time:
        where_clause = f"WHERE TRY_CAST({time_col} AS BIGINT) IS NOT NULL"

    con.execute(f"""
        CREATE OR REPLACE TEMP VIEW _clean AS
        SELECT *, TRY_CAST({time_col} AS BIGINT) AS ms
        FROM _raw
        {where_clause}
    """)
    mn, mx, n = con.execute("SELECT MIN(ms), MAX(ms), COUNT(*) FROM _clean").fetchone()
    if n == 0:
        raise ValueError("All timestamps are NaN after parsing; nothing to split.")

    if cutoff is None:
        q = 1.0 - float(test_fraction)
        cutoff_ms = int(con.execute("SELECT quantile_disc(ms, ?) FROM _clean", [q]).fetchone()[0])
    else:
        if isinstance(cutoff, (int, float)) and np.isfinite(cutoff):
            cutoff_ms = int(cutoff)
        else:
            cutoff_ms = int(pd.to_datetime(cutoff, utc=True).value // 1_000_000)

    if not (mn <= cutoff_ms <= mx):
        raise RuntimeError(f"Cutoff {cutoff_ms} outside data range [{mn}, {mx}].")

    op_train = "<=" if train_includes_cutoff else "<"
    op_test  = ">"  if train_includes_cutoff else ">="

    con.execute(f"CREATE OR REPLACE TEMP VIEW train AS SELECT * FROM _clean WHERE ms {op_train} {cutoff_ms}")
    con.execute(f"CREATE OR REPLACE TEMP VIEW test  AS SELECT * FROM _clean WHERE ms {op_test}  {cutoff_ms}")

    train_n = con.execute("SELECT COUNT(*) FROM train").fetchone()[0]
    test_n  = con.execute("SELECT COUNT(*) FROM test").fetchone()[0]
    if train_n == 0 or test_n == 0:
        raise RuntimeError(f"Empty split: train={train_n}, test={test_n}. Adjust `test_fraction`/`cutoff`.")

    if out_prefix:
        train_query = "SELECT * FROM train ORDER BY ms" if sort_within_splits else "SELECT * FROM train"
        test_query  = "SELECT * FROM test  ORDER BY ms" if sort_within_splits else "SELECT * FROM test"

        con.execute(f"COPY ({train_query}) TO '{out_prefix}/reviews_small_train' (FORMAT PARQUET)")
        con.execute(f"COPY ({test_query})  TO '{out_prefix}/reviews_small_test'  (FORMAT PARQUET)")


    cutoff_ts = pd.to_datetime(cutoff_ms, unit="ms", utc=True).tz_convert(None)
    return cutoff_ms, cutoff_ts, train_n, test_n

In [ ]:
# Create the output directory if it doesn't exist
output_dir = '/content/artifacts/Working/Post_temporal_splits'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

cutoff_ms, cutoff_ts, n_tr, n_te = temporal_split_ms_duckdb(
    '/content/artifacts/Working/reviews_filtered_by_items',
    time_col='timestamp',
    test_fraction=0.20,
    # cutoff = "01-01-2022",
    train_includes_cutoff=True,
    sort_within_splits=True,
    out_prefix=output_dir
)

print(f"Temporal split complete. Cutoff date: {cutoff_ts}")
print(f"Train split rows: {n_tr:,}")
print(f"Test split rows: {n_te:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Temporal split complete. Cutoff date: 2021-06-25 11:52:13.539000
Train split rows: 11,384,437
Test split rows: 2,846,109


In [ ]:
SAVE_DIR = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_temporal_splits'
os.makedirs(SAVE_DIR, exist_ok=True)

rev_g_train = f'{SAVE_DIR}/reviews_small_train'
rev_g_test = f'{SAVE_DIR}/reviews_small_test'

REV_SPLIT_TRAIN = "/content/artifacts/Working/Post_temporal_splits/reviews_small_train"
REV_SPLIT_TEST = "/content/artifacts/Working/Post_temporal_splits/reviews_small_test"

sql_copy_reviews_train = f"""
COPY (SELECT * FROM read_parquet('{REV_SPLIT_TRAIN}'))
TO '{rev_g_train}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_reviews_train)

sql_copy_reviews_test = f"""
COPY (SELECT * FROM read_parquet('{REV_SPLIT_TEST}'))
TO '{rev_g_test}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_reviews_test)

# quick check
import os
print('reviews_train:', os.path.getsize(rev_g_train)/1024**2, 'MB')
print('items_test  :', os.path.getsize(rev_g_test)/1024**2, 'MB')

reviews_train: 492.9410934448242 MB
items_test  : 124.59007453918457 MB


#### Iterative k-core filtering

In [ ]:
def kcore_filter_iterative_duckdb(
    src_parquet: str,
    *,
    user_col: str = "user_id",
    item_col: str = "parent_asin",
    user_k: int = 5,
    item_k: int = 5,
    max_iters: int = 100,
    out_path: Optional[str] = "/content/splits/kcore_train",
    return_history: bool = True,
    return_df: bool = False,
    verbose: bool = False,
) -> Tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    def q(ident: str) -> str:
        if '"' in ident:
            raise ValueError(f'Identifier {ident!r} contains a double quote (").')
        return f'"{ident}"'
    u, i = q(user_col), q(item_col)

    con = duckdb.connect()
    con.execute("PRAGMA disable_progress_bar")
    src = src_parquet.rstrip('/')

    # Start with a TABLE (materialized), not a view
    con.execute(f"CREATE OR REPLACE TEMP TABLE cur AS SELECT * FROM read_parquet('{src}')")
    n0 = con.execute("SELECT COUNT(*) FROM cur").fetchone()[0]
    if n0 == 0:
        raise ValueError("No rows to process.")

    history: List[Dict] = []

    for it in range(1, max_iters + 1):
        if verbose: print(f"Iteration {it} started")
        n_before = con.execute("SELECT COUNT(*) FROM cur").fetchone()[0]

        # User prune
        con.execute("DROP TABLE IF EXISTS ucnt")
        con.execute(f"CREATE TEMP TABLE ucnt AS SELECT {u} AS u, COUNT(*) AS c FROM cur GROUP BY {u}")
        con.execute("DROP TABLE IF EXISTS cur_u")
        con.execute(f"""
            CREATE TEMP TABLE cur_u AS
            SELECT c.* FROM cur c
            JOIN ucnt u ON c.{user_col} = u.u
            WHERE u.c >= {user_k}
        """)
        n_u = con.execute("SELECT COUNT(*) FROM cur_u").fetchone()[0]
        if n_u == 0:
            raise RuntimeError(f"All rows pruned at user step (iter={it}). Lower user_k/item_k.")

        # Item prune
        con.execute("DROP TABLE IF EXISTS icnt")
        con.execute(f"CREATE TEMP TABLE icnt AS SELECT {i} AS v, COUNT(*) AS c FROM cur_u GROUP BY {i}")
        con.execute("DROP TABLE IF EXISTS nxt")
        con.execute(f"""
            CREATE TEMP TABLE nxt AS
            SELECT u.* FROM cur_u u
            JOIN icnt v ON u.{item_col} = v.v
            WHERE v.c >= {item_k}
        """)
        n_after = con.execute("SELECT COUNT(*) FROM nxt").fetchone()[0]
        if n_after == 0:
            raise RuntimeError(f"All rows pruned at item step (iter={it}). Lower user_k/item_k.")

        users_after, items_after = con.execute(f"""
            SELECT COUNT(DISTINCT {u}), COUNT(DISTINCT {i}) FROM nxt
        """).fetchone()

        history.append({
            "iter": it,
            "rows_before": n_before,
            "rows_after": n_after,
            "users_after": users_after,
            "items_after": items_after,
            "removed": n_before - n_after,
        })

        # Convergence: no change this iteration
        if n_after == n_before:
            con.execute("DROP TABLE IF EXISTS cur")
            con.execute("CREATE TEMP TABLE cur AS SELECT * FROM nxt")
            if verbose: print(f"Iteration {it} finished (converged).")
            break

        # Prepare next iteration: replace cur with nxt (tables, so no cycles)
        con.execute("DROP TABLE IF EXISTS cur")
        con.execute("CREATE TEMP TABLE cur AS SELECT * FROM nxt")

    # Persist and/or return
    if out_path:
        con.execute(f"COPY (SELECT * FROM cur) TO '{out_path}' (FORMAT PARQUET)")
    out_df = con.execute("SELECT * FROM cur").df() if return_df else None
    hist_df = pd.DataFrame(history) if return_history else None
    return out_df, hist_df

In [ ]:
SAVE_DIR = '/content/artifacts/Working/Post_k_core_filtering'
os.makedirs(SAVE_DIR, exist_ok=True)
user_k = 5
item_k = 5
out_path = f'{SAVE_DIR}/reviews_small_train_{user_k}_{item_k}'

filtered_df, history = kcore_filter_iterative_duckdb(
    src_parquet='/content/artifacts/Working/Post_temporal_splits/reviews_small_train',  # or a folder with *.parquet
    user_col='user_id',
    item_col='parent_asin',
    user_k=user_k,
    item_k=item_k,
    max_iters=20,            # your data is already deduped earlier
    out_path=out_path,
    return_history=True,
    verbose=True,
    return_df=False                   # avoid pulling huge data back to RAM
)

Iteration 1 started
Iteration 2 started
Iteration 3 started
Iteration 4 started
Iteration 5 started
Iteration 6 started
Iteration 7 started
Iteration 8 started
Iteration 9 started
Iteration 10 started
Iteration 11 started
Iteration 11 finished (converged).


In [ ]:
history

,iter,rows_before,rows_after,users_after,items_after,removed
0,1,11384437,1171447,264366,53267,10212990
1,2,1171447,638062,106183,30716,533385
2,3,638062,547968,86184,26631,90094
3,4,547968,523998,81190,25520,23970
4,5,523998,516822,79705,25199,7176
5,6,516822,514757,79280,25107,2065
6,7,514757,513985,79132,25062,772
7,8,513985,513689,79070,25050,296
8,9,513689,513601,79050,25048,88
9,10,513601,513597,79049,25048,4


In [ ]:
SAVE_DIR = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_k_core_filtering'
os.makedirs(SAVE_DIR, exist_ok=True)

rev_g_train = f'{SAVE_DIR}/reviews_small_train_{user_k}_{item_k}'

REV_K_CORE_TRAIN = f"/content/artifacts/Working/Post_k_core_filtering/reviews_small_train_{user_k}_{item_k}"

sql_copy_reviews = f"""
COPY (SELECT * FROM read_parquet('{REV_K_CORE_TRAIN}'))
TO '{rev_g_train}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_reviews)

# quick check
import os
print('reviews_after_k_core_filtering:', os.path.getsize(rev_g_train)/1024**2, 'MB')

reviews_after_k_core_filtering: 21.00367546081543 MB


# Content Based Recommender system

In [5]:
SAVE_DIR_DRIVE = '/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/Post_items_cleanup'
ITM_G = f'{SAVE_DIR_DRIVE}/items_cleaned'

SAVE_DIR_LOCAL = '/content/artifacts/Working'
ITM_LOCAL = f'{SAVE_DIR_LOCAL}/items_cleaned'
os.makedirs(SAVE_DIR_LOCAL, exist_ok=True)


con = duckdb.connect()
con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='40GB';")  # optional

# Directly copy the data from Google Drive to the local directory using DuckDB
# This avoids loading the entire dataset into a pandas DataFrame
sql_copy_items = f"""
COPY (SELECT * FROM read_parquet('{ITM_G}'))
TO '{ITM_LOCAL}' (FORMAT PARQUET, COMPRESSION SNAPPY);
"""
con.execute(sql_copy_items)

# Check the size of the copied file
size_bytes = os.path.getsize(ITM_LOCAL)

def human(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024:
            return f"{n:.2f} {u}"
        n /= 1024
    return f"{n:.2f} PB"

print(f"Copied data to {ITM_LOCAL}. Size: {human(size_bytes)}")

# You can still load a small sample into a DataFrame for inspection if needed
# items_df = con.execute(f"SELECT * FROM read_parquet('{ITM_LOCAL}') LIMIT 10").fetchdf()
# print("\nSample of the copied data:")
# display(items_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Copied data to /content/artifacts/Working/items_cleaned. Size: 1.19 GB


## Basic Analysis of data

In [ ]:
print("Counts of each gender within each main category:")
print(items_df.groupby('main_categories')['gender'].value_counts())

Counts of each gender within each main category:
main_categories  gender      
clothing         womens          415265
                 mens            288071
                 girls            45068
                 boys             36764
                 unisex-adult     22404
                 baby-girls       21523
                 baby-boys        13618
                 unisex-child      9039
                 unisex-baby       2222
jewelry          womens          284340
                 mens             35051
                 unisex-adult     16047
                 boys             13795
                 girls             8326
                 unisex-child      6358
                 baby-girls          83
                 baby-boys           64
                 unisex-baby         20
shoes            womens          488019
                 mens            234279
                 girls            29596
                 unisex-child     23601
                 boys             19509
 

In [ ]:
items_df.columns

Index(['title', 'average_rating', 'rating_number', 'features', 'description',
       'price', 'categories', 'details', 'parent_asin', 'main_image_url',
       'gender', 'dimension_weight', 'manufacturer', 'main_categories',
       'date_first_available', 'leaf_category'],
      dtype='object')

In [ ]:
items_df["gender"].value_counts()

,count
gender,
womens,1187624
mens,557401
girls,82990
boys,70068
unisex-adult,49694
unisex-child,38998
baby-girls,25708
baby-boys,17123
unisex-baby,3184


## Downloading Images to Google Drive

In [73]:
# ───────────────────────── config ─────────────────────────
ITEMS_PARQUET   = ITM_LOCAL   # must have: parent_asin, main_image_url, main_categories, gender
BASE_DIR        = "/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing"

FINAL_TAR_ROOT        = f"{BASE_DIR}/images_tars"         # where .tar shards go
TMP_TAR_ROOT  = "/content/tars_tmp"                       # fast local
MANIFEST_PATH   = f"{BASE_DIR}/images_manifest.csv.gz"    # gzipped CSV

UA                = "Mozilla/5.0 (compatible; recsys-downloader/2.0)"
MAX_RETRIES       = 3
MAX_WORKERS       = 64             # Drive is happier with modest concurrency
MAX_PER_HOST      = 25             # domain throttling: concurrent requests per host
BATCH_ROWS        = 50_000         # DuckDB -> Arrow batch size
FUTURES_IN_FLIGHT = 1_000          # keep in-flight futures bounded

# tune these as you like
CONNECT_TIMEOUT = 3.0
READ_TIMEOUT    = 20.0
POOL_CONNS      = 64           # per scheme
POOL_MAXSIZE    = 128          # concurrent pooled connections per host
RETRY_TOTAL     = 3
RETRY_BACKOFF   = 0.3
RETRY_STATUSES  = (429, 500, 502, 503, 504)
HEADERS = {"User-Agent": UA}   # reuse your UA string

# per-tar limits (WebDataset style)
MAX_IMAGES_PER_TAR = 15_000
MAX_TAR_BYTES      = 4_000_000_000  # ~4GB safety cap

Path(FINAL_TAR_ROOT).mkdir(parents=True, exist_ok=True)

In [74]:
# ───────────────────── shard mapping (no leaf category) ─────────────────────
def compute_shard(df):
    g  = df["gender"].astype(str)
    mc = df["main_categories"].astype(str)

    cond1 = g.isin(["womens","mens"])
    cond2 = g.isin(["girls","boys","unisex-adult","unisex-child"])
    cond3 = g.isin(["baby-girls","baby-boys","unisex-baby"])

    # Convert np.nan to string 'nan' to avoid DTypePromotionError
    shard = np.where(cond1, mc + "|" + g,
             np.where(cond2, g,
             np.where(cond3, "baby", "nan")))
    df["shard"] = shard
    return df[df["shard"] != "nan"].copy() # Filter out rows where shard is 'nan' and return a copy

In [75]:
# ───────────────────── per-host concurrency control ─────────────────────
_host_semaphores = {}
_host_lock = threading.Lock()

def host_semaphore(host: str) -> threading.Semaphore:
    with _host_lock:
        sem = _host_semaphores.get(host)
        if sem is None:
            sem = threading.Semaphore(MAX_PER_HOST)
            _host_semaphores[host] = sem
        return sem


In [76]:
class TarShardManager:
    def __init__(self, tmp_root, final_root, max_items, max_bytes, name_fmt="{shard}-{seq:05d}.tar"):
        self.tmp_root   = Path(tmp_root)
        self.final_root = Path(final_root)
        self.max_items  = int(max_items)
        self.max_bytes  = int(max_bytes)
        self.name_fmt   = name_fmt
        self.tmp_root.mkdir(parents=True, exist_ok=True)
        self.final_root.mkdir(parents=True, exist_ok=True)
        self.open = {}  # shard -> meta

    def _next_seq_for_shard(self, shard: str) -> int:
        final_dir = self.final_root / shard
        if not final_dir.exists():
            return 0
        seq_re = re.compile(rf'^{re.escape(shard)}-(\d+)\.tar$')
        max_seq = -1
        for p in final_dir.iterdir():
            if not p.is_file():
                continue
            m = seq_re.match(p.name)
            if m:
                try:
                    s = int(m.group(1))
                    if s > max_seq: max_seq = s
                except ValueError:
                    pass
        return max_seq + 1  # start at 0 if none

    def _open_next(self, shard: str):
        meta = self.open.get(shard)
        next_seq = self._next_seq_for_shard(shard) if meta is None else meta["seq"] + 1

        tmp_dir   = self.tmp_root   / shard
        final_dir = self.final_root / shard
        tmp_dir.mkdir(parents=True, exist_ok=True)
        final_dir.mkdir(parents=True, exist_ok=True)

        # pick a seq that doesn't collide in FINAL
        fname      = self.name_fmt.format(shard=shard, seq=next_seq)
        final_path = final_dir / fname
        while final_path.exists():
            next_seq += 1
            fname      = self.name_fmt.format(shard=shard, seq=next_seq)
            final_path = final_dir / fname

        tmp_path = tmp_dir / (fname + ".part")
        tf = tarfile.open(tmp_path, mode="w")
        self.open[shard] = {
            "tar": tf, "count": 0, "seq": next_seq, "bytes": 0,
            "tmp_path": tmp_path, "final_dir": final_dir
        }

    def _finalize_current(self, shard: str, reopen: bool):
        meta = self.open.get(shard)
        if not meta or meta["tar"] is None:
            return None
        meta["tar"].close()

        fname      = self.name_fmt.format(shard=shard, seq=meta["seq"])
        final_path = meta["final_dir"] / fname

        # last-resort guard: never overwrite if somehow present
        if final_path.exists():
            new_seq = meta["seq"] + 1
            while (meta["final_dir"] / self.name_fmt.format(shard=shard, seq=new_seq)).exists():
                new_seq += 1
            final_path = meta["final_dir"] / self.name_fmt.format(shard=shard, seq=new_seq)

        shutil.move(str(meta["tmp_path"]), str(final_path))
        if reopen:
            self._open_next(shard)
        else:
            self.open[shard] = {"tar": None, "count": 0, "seq": meta["seq"],
                                "bytes": 0, "tmp_path": None, "final_dir": meta["final_dir"]}
        return str(final_path)

    def add(self, shard: str, member_path: str, data_bytes: bytes) -> str:
        if shard not in self.open or self.open[shard]["tar"] is None:
            self._open_next(shard)
        meta = self.open[shard]
        if meta["count"] >= self.max_items or meta["bytes"] >= self.max_bytes:
            self._finalize_current(shard, reopen=True)
            meta = self.open[shard]

        info       = tarfile.TarInfo(name=member_path)
        info.size  = len(data_bytes)
        info.mtime = int(time.time())
        meta["tar"].addfile(info, io.BytesIO(data_bytes))
        meta["count"] += 1
        meta["bytes"] += len(data_bytes)

        final_path = meta["final_dir"] / self.name_fmt.format(shard=shard, seq=meta["seq"])
        return str(final_path)

    def close_all(self, delete_tmp: bool = True):
        for shard in list(self.open.keys()):
            self._finalize_current(shard, reopen=False)
        self.open.clear()
        if delete_tmp:
            shutil.rmtree(self.tmp_root, ignore_errors=True)

In [77]:
# ---------- pooled, thread-local Session ----------
_THREAD_LOCAL = threading.local()

def get_session() -> requests.Session:
    s = getattr(_THREAD_LOCAL, "session", None)
    if s is None:
        s = requests.Session()
        retry = Retry(
            total=RETRY_TOTAL,
            backoff_factor=RETRY_BACKOFF,
            status_forcelist=RETRY_STATUSES,
            allowed_methods=frozenset(["GET", "HEAD"]),
            raise_on_status=False,
        )
        adapter = HTTPAdapter(
            pool_connections=POOL_CONNS,
            pool_maxsize=POOL_MAXSIZE,
            max_retries=retry,
        )
        s.mount("http://",  adapter)
        s.mount("https://", adapter)
        _THREAD_LOCAL.session = s
    return s

In [78]:
# ───────────────────── http fetch → jpeg bytes ─────────────────────
def fetch_to_jpeg(asin, url, shard):
    """Return tuple for manifest: (asin,url,shard,ok,http_status,error,attempts,data_bytes)"""
    attempts = 0
    status   = None
    errtxt   = ""
    data     = None
    host     = urlparse(url).netloc or "unknown"
    sem      = host_semaphore(host)
    sess = get_session()

    for attempts in range(1, MAX_RETRIES+1):
        try:
            with sem:  # cap concurrency per host
                r = sess.get(
                    url,
                    headers=HEADERS,
                    timeout=(CONNECT_TIMEOUT, READ_TIMEOUT),
                    stream=False,   # full read into memory
                )
            status = r.status_code
            if status == 200 and r.content:
                try:
                    with Image.open(io.BytesIO(r.content)) as im:
                        im = im.convert("RGB")
                        buf = io.BytesIO()
                        im.save(buf, format="JPEG", quality=90, optimize=True)
                        data = buf.getvalue()
                    errtxt = ""
                    break
                except UnidentifiedImageError as e:
                    errtxt = f"pil_decode:{e.__class__.__name__}"
                except Exception as e:
                    errtxt = f"pil_other:{e.__class__.__name__}"
            else:
                errtxt = f"http_status:{status}"
        except requests.exceptions.Timeout:
            errtxt = "http_timeout"
        except requests.exceptions.ConnectionError:
            errtxt = "http_conn"
        except Exception as e:
            errtxt = f"http_other:{e.__class__.__name__}"

        time.sleep(0.3 * attempts)  # simple backoff

    ok = data is not None
    if ok:
        errtxt = ""
    return asin, url, shard, ok, status or "", errtxt, attempts, data or b""

In [79]:
# ───────────────────── manifest writer (streaming, resume-aware) ─────────────────────
class ManifestWriter:
    def __init__(self, path):
        self.path = path
        # append mode; create if missing
        exists = os.path.exists(self.path)
        self.fh = gzip.open(self.path, "at", newline="")
        self.w  = csv.writer(self.fh)
        if not exists or os.stat(self.path).st_size == 0:
            self.w.writerow([
                "parent_asin","url","shard","hash_prefix",
                "tar_path","tar_member","ok","http_status","error","attempts","bytes"
            ])
        self.count = 0

    def write(self, row):
        self.w.writerow(row)
        self.count += 1
        if self.count % 5000 == 0:
            self.fh.flush()

    def close(self):
        try:
            self.fh.flush()
            self.fh.close()
        except Exception:
            pass


In [80]:
def repair_tmp(tmp_root: str, final_root: str):
    """Move any stale *.tar.part from tmp_root to final_root/<shard>/<fname>.tar."""
    tmp_root_p  = Path(tmp_root)
    final_root_p= Path(final_root)
    if not tmp_root_p.exists():
        return
    for part in tmp_root_p.rglob("*.tar.part"):
        try:
            shard = part.parent.name
            final_dir = final_root_p / shard
            final_dir.mkdir(parents=True, exist_ok=True)
            final_name = part.name.replace(".tar.part", ".tar")
            shutil.move(str(part), str(final_dir / final_name))
        except Exception:
            # best effort: skip corrupted/locked files
            pass

In [88]:
from logging import raiseExceptions
# ───────────────────── driver with resume ─────────────────────
def run():

    # If a manifest exists, build a temp view of successfully downloaded rows
    if os.path.exists(MANIFEST_PATH):
        con.execute(f"""
            CREATE OR REPLACE TEMP VIEW man_ok AS
            SELECT parent_asin, url
            FROM read_csv_auto('{MANIFEST_PATH}', union_by_name=True)
            WHERE ok = 1
        """)
        resume_predicate = """
          AND NOT EXISTS (
              SELECT 1 FROM man_ok m
              WHERE m.parent_asin = i.parent_asin
                AND m.url        = i.main_image_url
          )
        """
        print("Resume: existing manifest found; rows with ok=1 will be skipped.")
    else:
        resume_predicate = ""  # nothing to skip
        print("Fresh run: no manifest found; downloading all rows.")

    # 1) Compute total to download (AFTER resume filter)
    total_remaining = con.execute(f"""
        SELECT COUNT(*) FROM (
          SELECT 1
          FROM read_parquet('{ITEMS_PARQUET}') i
          WHERE i.parent_asin IS NOT NULL
            AND i.main_image_url IS NOT NULL
            AND i.main_categories IS NOT NULL
            AND i.gender IS NOT NULL
            {resume_predicate}
        ) t
    """).fetchone()[0]

    print(f"Total remaining to download: {total_remaining:,}")

    # Stream items to download (skipping those already ok==1)
    cur = con.execute(f"""
        SELECT i.parent_asin, i.main_image_url, i.main_categories, i.gender
        FROM read_parquet('{ITEMS_PARQUET}') i
        WHERE i.parent_asin IS NOT NULL
          AND i.main_image_url IS NOT NULL
          AND i.main_categories IS NOT NULL
          AND i.gender IS NOT NULL
          {resume_predicate}
    """)
    reader = cur.fetch_record_batch(rows_per_batch=BATCH_ROWS)

    tars = TarShardManager(
        tmp_root=TMP_TAR_ROOT,
        final_root=FINAL_TAR_ROOT,
        max_items=MAX_IMAGES_PER_TAR,
        max_bytes=MAX_TAR_BYTES
    )
    mani  = ManifestWriter(MANIFEST_PATH)

    processed = 0
    ok_count  = 0
    pbar = tqdm(total=total_remaining, unit="img", desc="Downloading → tars")

    try:

        while True:
            try:
                batch = reader.read_next_batch()
            except StopIteration:
                break  # newer Arrow: signals end via exception

            if batch is None or batch.num_rows == 0:
                break

            df = batch.to_pandas(types_mapper=None)
            df = compute_shard(df)

            # Create job tuples (asin, url, shard)
            jobs_iter = (
                (asin, url, shard)
                for asin, url, shard in df[["parent_asin","main_image_url","shard"]].itertuples(index=False, name=None)
            )

            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
                inflight = set()

                def submit_some(n):
                    for _ in range(n):
                        try:
                            a, u, s = next(jobs_iter)
                        except StopIteration:
                            return False
                        inflight.add(ex.submit(fetch_to_jpeg, a, u, s))
                    return True

                submit_some(min(FUTURES_IN_FLIGHT, MAX_WORKERS * 16))

                while inflight:
                    for fut in as_completed(inflight):
                        inflight.remove(fut)
                        asin, url, shard, ok, status, errtxt, attempts, data = fut.result()
                        processed += 1

                        if ok:
                            h = hashlib.md5(url.encode("utf-8")).hexdigest()[:10]
                            prefix = h[:2]
                            member = f"{prefix}/{asin}_{h}.jpg"
                            tar_path = tars.add(shard, member, data)
                            mani.write([asin, url, shard, prefix, tar_path, member, 1, status, "", attempts, len(data)])
                            ok_count += 1
                        else:
                            mani.write([asin, url, shard, "", "", "", 0, status, errtxt, attempts, 0])

                        # top up queue
                        submit_some(1)

                        # Progress shows processed & ok
                        pbar.set_postfix_str(f"ok={ok_count:,}")
                        pbar.update(1)

    finally:
        pbar.close()
        mani.close()
        tars.close_all(delete_tmp=True)
        print("\nEven if the system might have crashed the remaining tars_temp file was exported to Google drive safely. If it's a successful run then Congratulations buddy!!!😎")

    print(f"\nDone. Processed={processed:,}, ok={ok_count:,}.")
    print(f"Manifest: {MANIFEST_PATH}")
    print(f"Tar shards root: {FINAL_TAR_ROOT}")

repair_tmp(TMP_TAR_ROOT, FINAL_TAR_ROOT)
run()

Resume: existing manifest found; rows with ok=1 will be skipped.
Total remaining to download: 1,380,125


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Even if the system might have crashed the remaining tars_temp file was exported to Google drive safely. If it's a successful run then Congratulations buddy!!!😎

Done. Processed=1,380,125, ok=1,380,027.
Manifest: /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_manifest.csv.gz
Tar shards root: /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_tars


In [89]:
import pandas as pd
import gzip

# Define the path to the manifest file
MANIFEST_PATH = f"/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images_manifest.csv.gz"

# Read the gzipped CSV file into a pandas DataFrame
try:
    with gzip.open(MANIFEST_PATH, 'rt') as f:
        manifest_df = pd.read_csv(f)

    # Display the first few rows of the DataFrame
    print("Manifest file content:")
    display(manifest_df.head())

except FileNotFoundError:
    print(f"Error: Manifest file not found at {MANIFEST_PATH}")
except Exception as e:
    print(f"An error occurred while reading the manifest file: {e}")

/tmp/ipython-input-3953762423.py:10: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  manifest_df = pd.read_csv(f)


Manifest file content:


,parent_asin,url,shard,hash_prefix,tar_path,tar_member,ok,http_status,error,attempts,bytes
0,B00M44BSAG,https://m.media-amazon.com/images/I/514qGTrvTv...,clothing|womens,a2,/content/drive/MyDrive/Product_Recommender_End...,a2/B00M44BSAG_a25dd49db5.jpg,1,200,NaN,1,30546
1,B00E1HICEY,https://m.media-amazon.com/images/I/81P6eVm5Gy...,clothing|mens,76,/content/drive/MyDrive/Product_Recommender_End...,76/B00E1HICEY_76da1adf8e.jpg,1,200,NaN,1,199763
2,B00J62LNCW,https://m.media-amazon.com/images/I/81615aGbaQ...,shoes|mens,5e,/content/drive/MyDrive/Product_Recommender_End...,5e/B00J62LNCW_5e1743c97a.jpg,1,200,NaN,1,195485
3,B096Z2SMWT,https://m.media-amazon.com/images/I/71OvgY2Bh4...,unisex-child,03,/content/drive/MyDrive/Product_Recommender_End...,03/B096Z2SMWT_03ac9c4e34.jpg,1,200,NaN,1,251707
4,B075NXTTJW,https://m.media-amazon.com/images/I/61K1RxY2rR...,shoes|womens,ec,/content/drive/MyDrive/Product_Recommender_End...,ec/B075NXTTJW_ecd037bc0f.jpg,1,200,NaN,1,104278


In [90]:
manifest_df.shape

(2032836, 11)

In [91]:
manifest_df[manifest_df.ok == 0].head()

,parent_asin,url,shard,hash_prefix,tar_path,tar_member,ok,http_status,error,attempts,bytes
13727,B0016GZQUG,https://m.media-amazon.com/images/I/61W0Me1DsK...,jewelry|womens,NaN,NaN,NaN,0,404,http_status:404,3,0
50643,B0062S0NSS,https://m.media-amazon.com/images/I/51X7PfarDA...,girls,NaN,NaN,NaN,0,404,http_status:404,3,0
60999,B000UWVMVE,https://m.media-amazon.com/images/I/71BRrUmBHS...,shoes|womens,NaN,NaN,NaN,0,404,http_status:404,3,0
63651,B000FBRTRQ,https://m.media-amazon.com/images/I/716z69ip72...,shoes|mens,NaN,NaN,NaN,0,404,http_status:404,3,0
74997,B09DLQ3YVB,https://m.media-amazon.com/images/I/81jPy1DsTy...,clothing|womens,NaN,NaN,NaN,0,404,http_status:404,3,0


In [92]:
print(manifest_df[manifest_df.ok == 0]["url"])

13727      https://m.media-amazon.com/images/I/61W0Me1DsK...
50643      https://m.media-amazon.com/images/I/51X7PfarDA...
60999      https://m.media-amazon.com/images/I/71BRrUmBHS...
63651      https://m.media-amazon.com/images/I/716z69ip72...
74997      https://m.media-amazon.com/images/I/81jPy1DsTy...
                                 ...                        
1872717    https://m.media-amazon.com/images/I/71gB6vvDlo...
1874296    https://m.media-amazon.com/images/I/81z6SH-mfH...
1911237    https://m.media-amazon.com/images/I/81DigD6nsD...
1912996    https://m.media-amazon.com/images/I/91fHeJVKuf...
1933867    https://m.media-amazon.com/images/I/71UwsypjEH...
Name: url, Length: 144, dtype: object


## Old Method of downloading

In [ ]:
ITEMS_PARQUET   = ITM_LOCAL  # must have parent_asin, main_image_url
BASE_DIR        = "/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing"         # where to store images & features

IMG_DIR         = f"{BASE_DIR}/images"
FEAT_DIR        = f"{BASE_DIR}/content-based/Resnet/features"

TIMEOUT         = 15
MAX_RETRIES     = 3

# ─────────────────────────── helpers: io ────────────────────────────
def safe_mkdir(p):
    Path(p).mkdir(parents=True, exist_ok=True)
safe_mkdir(IMG_DIR)
safe_mkdir(FEAT_DIR)

In [ ]:
con = duckdb.connect()
items = con.execute(f"""
    SELECT parent_asin, main_image_url
    FROM read_parquet('{ITEMS_PARQUET}')
    WHERE main_image_url IS NOT NULL AND parent_asin IS NOT NULL
""").fetchdf()

print(f"Items with images: {len(items):,}")

# ─────────────────────── image download + cache ─────────────────────
UA           = "Mozilla/5.0 (compatible; recsys-resnet/1.0)"
TIMEOUT      = 15
MAX_RETRIES  = 3
MAX_WORKERS  = 32
BATCH_SUBMIT = 5_000  # futures in-flight per batch

def url_to_path(url: str, asin: str) -> str:
    h = hashlib.md5(url.encode("utf-8")).hexdigest()[:10]
    return os.path.join(IMG_DIR, f"{asin}_{h}.jpg")

def fetch_image(asin: str, url: str):
    out = url_to_path(url, asin)
    if os.path.exists(out):
        return asin, out, True
    ok = False
    for _ in range(MAX_RETRIES):
        try:
            r = requests.get(url, headers={"User-Agent": UA}, timeout=TIMEOUT)
            if r.status_code == 200 and r.content:
                with Image.open(io.BytesIO(r.content)) as im:
                    im = im.convert("RGB")
                    im.save(out, format="JPEG", quality=90, optimize=True)
                ok = True
                break
        except Exception:
            pass
        time.sleep(0.3)
    return asin, out, ok

def row_iter(df):
    # fast, low-overhead iterator of (asin, url)
    for asin, url in df[['parent_asin','main_image_url']].itertuples(index=False, name=None):
        yield asin, url

st = time.time()
print("Downloading (or reusing cached) images...")

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex, tqdm(total=len(items), unit="img") as pbar:
    it = row_iter(items)
    while True:
        batch = list(islice(it, BATCH_SUBMIT))
        if not batch:
            break
        futures_batch = [ex.submit(fetch_image, a, u) for a, u in batch]
        for fut in as_completed(futures_batch):
            results.append(fut.result())
            pbar.update(1)

et = time.time()

dl_df = pd.DataFrame(results, columns=["parent_asin","img_path","ok"])
dl_df = dl_df[dl_df.ok]
print(f"Images ok: {len(dl_df):,} / {len(items):,}")
print(f"Time Elapsed: {et - st:.2f}s")

Items with images: 2,032,790


  0%|          | 0/2032790 [00:00<?, ?img/s]

In [ ]:
import os
import hashlib
ans = []


for i in range(50_000):
    # Take a random sample of 1 item
    random_item = items_df.iloc[i]

    # Get parent_asin and main_image_url from the random item
    random_asin = random_item['parent_asin']
    random_url = random_item['main_image_url']

    # Define the base directory for images
    a = "/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing"
    b = f"{a}/images"

    # Generate the expected image path
    def url_to_path(url: str, asin: str) -> str:
        h = hashlib.md5(url.encode("utf-8")).hexdigest()[:10]
        return os.path.join(b, f"{asin}_{h}.jpg")

    expected_image_path = url_to_path(random_url, random_asin)

    # Check if the image file exists
    image_exists = os.path.exists(expected_image_path)

    if image_exists:
        image_size = os.path.getsize(expected_image_path)
    else:
        image_size = None

    ans.append((image_exists, image_size))

In [ ]:
for i in ans:
  if not i[0]:
    print(i)

(False, None)


In [ ]:
!find /content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing/images -type f | wc -l

548621


In [ ]:
import duckdb, os, hashlib

CHUNK = 200_000

def url_to_path(url, asin):
    h = hashlib.md5(url.encode('utf-8')).hexdigest()[:10]
    return os.path.join(IMG_DIR, f"{asin}_{h}.jpg")

con = duckdb.connect()
cur = con.execute(f"""
SELECT parent_asin, main_image_url
FROM read_parquet('{ITEMS_PARQUET}')
WHERE parent_asin IS NOT NULL AND main_image_url IS NOT NULL
""")

reader = cur.fetch_record_batch(rows_per_batch=CHUNK)  # stream rows in batches

total = 0
have  = 0
while True:
    batch = reader.read_next_batch()
    if batch is None or batch.num_rows == 0:
        break
    df = batch.to_pandas(types_mapper=None)  # no object expansion surprises
    total += len(df)
    print(total)
    paths = (
        url_to_path(url, asin)
        for asin, url in df[['parent_asin','main_image_url']].itertuples(index=False, name=None)
    )
    have += sum(map(os.path.exists, paths))
    print(f"Images present so far: {have:,} / {total:,}")

print(f"FINAL images present: {have:,} / {total:,}")


200000
Images present so far: 199,995 / 200,000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

400000
Images present so far: 399,983 / 400,000
600000
Images present so far: 548,620 / 600,000
800000
Images present so far: 548,620 / 800,000
1000000
Images present so far: 548,620 / 1,000,000
1200000
Images present so far: 548,620 / 1,200,000
1400000
Images present so far: 548,620 / 1,400,000
1600000
Images present so far: 548,620 / 1,600,000
1800000
Images present so far: 548,620 / 1,800,000
2000000
Images present so far: 548,620 / 2,000,000
2032790
Images present so far: 548,620 / 2,032,790


StopIteration: 

In [ ]:
BASE_DIR        = "/content/drive/MyDrive/Product_Recommender_End_to_End_with_virtual_dressing"
IMG_DIR         = f"{BASE_DIR}/images"      # already downloaded
FEAT_DIR        = f"{BASE_DIR}/content-based/Resnet/features"  # will write here\
ITEMS_PARQUET   = ITM_LOCAL

DFA_COL   = "date_first_available"                 # your column name for Date First Available
MC_COL    = "main_categories"     # clothing / shoes / jewelry
GENDER_COL= "gender"
BITSET_FREQ = "M"                 # monthly cumulative bitsets

Path(FEAT_DIR).mkdir(parents=True, exist_ok=True)

# ====== 1) LOAD ITEMS + JOIN WITH DOWNLOADED IMAGES ======
con = duckdb.connect()
items = con.execute(f"""
    SELECT parent_asin, main_image_url, {MC_COL} AS main_category,
           {GENDER_COL} AS gender,
           {DFA_COL} AS dfa
    FROM read_parquet('{ITEMS_PARQUET}')
    WHERE parent_asin IS NOT NULL AND main_image_url IS NOT NULL
""").fetchdf()

def url_to_path(url: str, asin: str) -> str:
    h = hashlib.md5(url.encode("utf-8")).hexdigest()[:10]
    return os.path.join(IMG_DIR, f"{asin}_{h}.jpg")

items["img_path"] = [url_to_path(u, a) for a, u in items[["parent_asin","main_image_url"]].itertuples(index=False, name=None)]
items = items[items["img_path"].map(os.path.exists)]
print(f"Images ok after join: {len(items):,}")

# parse DFA -> epoch days (int32); fill missing as 0 (very old)
dfa_ts = pd.to_datetime(items["dfa"], utc=True, errors="coerce")
items["dfa_day"] = (dfa_ts.view("int64") // 86_400_000_000_000).astype("Int64").fillna(0).astype(np.int32)

# ====== 2) SHARD KEY (no leaf-category splitting) ======
def map_shard(row):
    g = row["gender"]
    mc = row["main_category"]
    if g in ("womens", "mens"):
        return f"{mc}|{g}"                   # e.g., clothing|womens
    if g in ("girls", "boys", "unisex-adult", "unisex-child"):
        return g                             # global shard
    if g in ("baby-girls", "baby-boys", "unisex-baby"):
        return "baby"                        # merged baby shard
    return None

items["shard"] = items.apply(map_shard, axis=1)
# items = items.dropna(subset=["shard"])
print("Shard sizes:")
print(items["shard"].value_counts())

Images ok after join: 10,000
Shard sizes:
shard
shoes|womens       2691
clothing|womens    2032
shoes|mens         1447
clothing|mens      1432
jewelry|womens      978
unisex-child        326
girls               293
boys                284
unisex-adult        207
baby                187
jewelry|mens        123
Name: count, dtype: int64


/tmp/ipython-input-46350216.py:33: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  items["dfa_day"] = (dfa_ts.view("int64") // 86_400_000_000_000).astype("Int64").fillna(0).astype(np.int32)


In [ ]:
%whos

Variable                 Type                    Data/Info
----------------------------------------------------------
BASE_DIR                 str                     /content/drive/MyDrive/Pr<...>End_with_virtual_dressing
BITSET_FREQ              str                     M
DFA_COL                  str                     date_first_available
DataLoader               type                    <class 'torch.utils.data.dataloader.DataLoader'>
Dataset                  type                    <class 'torch.utils.data.dataset.Dataset'>
Dict                     _SpecialGenericAlias    typing.Dict
FEAT_DIR                 str                     /content/drive/MyDrive/Pr<...>ent-based/Resnet/features
GENDER_COL               str                     gender
HfApi                    type                    <class 'huggingface_hub.hf_api.HfApi'>
IMG_DIR                  str                     /content/drive/MyDrive/Pr<...>h_virtual_dressing/images
ITEMS_PARQUET            str                     /c

In [ ]:
# ====== 3) RESNET50 FEATURE EXTRACTION ======
device = "cuda" if torch.cuda.is_available() else "cpu"
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Identity()
model.eval().to(device)

preproc = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=models.ResNet50_Weights.IMAGENET1K_V2.meta["mean"],
                         std=models.ResNet50_Weights.IMAGENET1K_V2.meta["std"]),
])